# 04 — Structured Outputs and Typed Interfaces

You are the AI engineer responsible for an insurance claim-intake boundary. Build an extraction pipeline, measure its naive baseline, classify its failures, introduce typed and semantic validation, compare approaches, break the improved system, and decide whether it is safe to connect to a case queue.

## Scenario, experimental question, and success criteria

The operations team receives free-form claim, inquiry, and complaint messages. Downstream software needs a nested record with a contact union, date, optional money, evidence references, a bounded next action, and explicit missing fields.

**Question:** How much does constrained generation reduce parse and schema failures, and which semantic failures remain?

A strategy succeeds only when it preserves the labelled facts and makes the correct accept/reject decision. Schema validity alone is not the release criterion.

## Learning objectives and safety boundary

You will distinguish parse, schema, and semantic validation; implement manual and Pydantic validators; compare free text, prompt-only JSON, application validation, and provider-native structured output; inject failures; and map the notebook to a production boundary. All data are synthetic. This lab extracts a proposal—it never approves a claim or performs an external action.

## Environment and reproducibility

- Python 3.10+; dependencies come from the repository `requirements.txt`.
- Dataset: 20 synthetic cases with named slices.
- Default mode: `PROMPT_COURSE_PROVIDER=mock`; deterministic and credential-free.
- Live mode: export your own `OPENAI_API_KEY`, set `PROMPT_COURSE_PROVIDER=openai`, and restart the kernel. Never paste a key here.
- Expected offline runtime: under one minute. Latency below is measured with `time.perf_counter`; token metadata comes from the selected provider when available.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from dataclasses import asdict
from pathlib import Path
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('src').resolve()))
LAB_PATH = Path('curriculum/beginner/04-structured-outputs-and-typed-interfaces/lab.py')
spec = spec_from_file_location('course04_lab', LAB_PATH)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)

cases = lab.load_cases()
print({'python_mode': os.getenv('PROMPT_COURSE_PROVIDER', 'mock'), 'cases': len(cases), 'slices': sorted({c.slice for c in cases})})
assert len(cases) == 20

## Architecture: three different questions

`document → model candidate → JSON parse → schema validation → semantic/evidence validation → accept or route safely`

A parser asks whether the serialization is legal. A schema asks whether the object has the agreed shape and types. A semantic evaluator asks whether the valid values are supported by the source and business rules. Keep these stages separately observable so the team fixes the correct component.

In [ ]:
sample = cases[0]
print(sample.input)
print(sample.expected)
print(lab.CaseRecord.model_json_schema()['required'])

## Baseline: free text at a software boundary

The naive implementation asks for a concise summary. It may be pleasant prose, but downstream code has no reliable fields, types, or failure state. Run it across the complete suite rather than judging one output.

In [ ]:
baseline_results = [lab.evaluate_candidate(case, 'free_text') for case in cases]
baseline = pd.DataFrame([asdict(result) for result in baseline_results])
baseline[['case_id', 'slice', 'parse_success', 'schema_valid', 'semantic_correct', 'accepted']].head()

In [ ]:
baseline_metrics = {
    metric: baseline[metric].mean()
    for metric in ['parse_success', 'schema_valid', 'semantic_correct', 'safe_decision']
}
print(baseline_metrics)
assert baseline_metrics['parse_success'] == 0.0

## Inspect baseline failures

This is an **interface/schema** failure, not evidence that the prose is linguistically poor. Changing tone or persona cannot create a dependable machine boundary. The smallest relevant intervention is a structured contract followed by deterministic validation.

## Step 1 — prompt-only JSON

Asking for JSON improves parseability, but the deterministic fixture injects truncation, extra fields, invalid enums, missing nested data, and wrong-but-valid values. The naive consumer accepts any parsed object, making its hidden assumption visible.

In [ ]:
json_results = [lab.evaluate_candidate(case, 'json_prompt') for case in cases]
json_frame = pd.DataFrame([asdict(result) for result in json_results])
print(json_frame.groupby('error_category', dropna=False).size())
json_frame.loc[~json_frame.safe_decision, ['case_id', 'slice', 'error_category', 'accepted', 'semantic_correct']].head(8)

## Step 2 — implement the primitive manually

`manual_schema_errors` checks the exact root fields, critical enums, and a non-empty evidence list. This is intentionally readable. It also reveals the maintenance risk: a hand-written validator can omit nested contact, money, date, and evidence constraints.

In [ ]:
manual_results = [lab.evaluate_candidate(case, 'manual_schema') for case in cases]
manual_frame = pd.DataFrame([asdict(result) for result in manual_results])
manual_frame.groupby(['error_category', 'accepted']).size()

## Step 3 — Pydantic as a typed application boundary

Pydantic packages JSON parsing, nested validation, discriminated unions, patterns, enums, and forbidden extra fields. We still run `semantic_errors` after it. The framework reduces validator code; it does not replace evidence checks or authorization.

In [ ]:
pydantic_results = [lab.evaluate_candidate(case, 'pydantic_validation') for case in cases]
pydantic_frame = pd.DataFrame([asdict(result) for result in pydantic_results])
pydantic_frame.groupby(['slice', 'error_category'], dropna=False).size().unstack(fill_value=0)

## Run the full comparison

The provider-native offline fixture represents schema-constrained generation and deliberately introduces a schema-valid action error in selected cases. This teaches the central limit: constrained decoding targets form, not truth.

In [ ]:
all_results = lab.run_experiment()
summary = pd.DataFrame(lab.summarize(all_results)).set_index('strategy')
summary.round(4)

In [ ]:
metrics = ['parse_success', 'schema_valid', 'semantic_correct', 'safe_decision']
axis = summary[metrics].plot.bar(figsize=(11, 5), ylim=(0, 1.05), color=['#7A869A', '#2F6FED', '#18A36B', '#C44536'])
axis.set_title('Same 20 cases: validity and safety by strategy')
axis.set_ylabel('Rate')
axis.set_xlabel('Strategy')
axis.grid(axis='y', alpha=.25)
plt.xticks(rotation=25, ha='right')
plt.show()

## Interpret the comparison

Manual checks and Pydantic can reject malformed candidates, while a parser-only consumer accepts some wrong objects. Provider-native constraints remove the fixture's parse and shape errors, but semantic errors remain and must be caught after generation. Compare abstractions on portability, validation coverage, error quality, provider capability, and consumer compatibility—not lines of code alone.

## Optional live structured-output experiment

The next cell always runs. In mock mode it validates the labelled fixture through the same typed provider contract. With your own key and explicit OpenAI mode, it uses the Responses API Pydantic parser. Inspect mode, model, measured elapsed time, and usage provenance; do not compare live quality from one case.

In [ ]:
provider_result = lab.run_provider_case(cases[0])
print(provider_result.value.model_dump(mode='json'))
print({
    'mode': provider_result.response.mode,
    'provider': provider_result.response.provider,
    'model': provider_result.response.model,
    'elapsed_seconds': provider_result.response.elapsed_seconds,
    'usage': provider_result.response.usage,
})

## Failure injection — diagnose before repairing

Select the extra-field candidate. Its facts are otherwise usable, so one bounded structural repair may be justified. Contrast it with missing claimant data or a wrong amount: deleting an extra key is not equivalent to inventing evidence.

In [ ]:
extra_case = next(case for case in cases if int(case.id.split('-')[-1]) % 5 == 1)
raw = lab.candidate_for(extra_case, 'json_prompt')
before = lab.evaluate_candidate(extra_case, 'pydantic_validation', raw)
repaired = lab.bounded_repair(raw)
after = lab.evaluate_candidate(extra_case, 'pydantic_validation', repaired)
print({'before': before.error_category, 'after': after.error_category, 'safe_after': after.safe_decision})
assert before.error_category == 'schema' and after.semantic_correct

## Diagnose the failure

Use `PROMPT / CONTEXT / EXAMPLE / MODEL / SCHEMA / RETRIEVAL / TOOL / WORKFLOW / EVALUATOR / SECURITY / RUNTIME`. Truncated JSON may be runtime/output-limit or model behavior. Extra keys are schema/interface drift. A wrong amount can be context selection, extraction/model behavior, or a bad label/evaluator. An authorized effect is a security/workflow decision and must never be repaired with stronger prompt wording.

In [ ]:
result_frame = pd.DataFrame([asdict(result) for result in all_results])
slice_view = (result_frame.groupby(['strategy', 'slice'])[['schema_valid', 'semantic_correct', 'safe_decision']].mean().round(3))
slice_view.loc[['json_prompt', 'provider_native']]

## Production upgrade

| Notebook | Production |
| --- | --- |
| Local JSONL | governed, versioned labelled dataset |
| One Pydantic model | versioned schema and consumer compatibility tests |
| Sequential loop | bounded concurrency, timeouts and backpressure |
| Printed errors | privacy-aware structured traces and alerts |
| Synthetic evidence | authorized tenant-scoped document service |
| Developer key | secret manager or workload identity |
| Local acceptance | CI release gate, canary, rollback and schema migration |

Monitor parse/schema/semantic failure rates separately, plus refusal/incomplete states, token usage, latency distributions, repair attempts, and consumer errors. Keep model proposals separate from authorization and external effects.

## When not to use the approach

Prefer a form, deterministic parser, database query, or source API when it already provides reliable typed fields. Do not force genuinely absent information into required fields. Do not mistake schema-constrained generation for evidence validation, tenant isolation, permission checks, or claim approval.

## Review questions, exercises, and advanced challenge

1. Why can valid JSON be unsafe?
2. Which layer should reject an unknown enum? Which should reject the wrong amount?
3. What makes a repair bounded?
4. Why forbid extra fields at a downstream boundary?

**Exercise 1:** add an organization claimant variant and two labelled cases; predict affected validators first.

**Exercise 2:** require invoice evidence whenever an amount is present and report regressions by slice.

**Advanced challenge:** run the same held-out slice across two model snapshots available to your OpenAI project. Make a release decision using schema validity, semantic accuracy, provider token usage, measured latency, and refusal/incomplete states.

## Summary

A professional structured-output boundary layers constrained generation, parsing, typed validation, semantic/evidence validation, safe routing, and observability. The schema reduces one class of uncertainty; it does not prove truth or grant authority.